# Labour-share accounting decomposition

This notebook consumes the processed Eurostat decomposition panel. It does not reimplement the accounting logic. The identity is

$$\Delta\log w = \Delta\log Y + \Delta\log s_L - \Delta\log N + (\pi_Y-\pi_C).$$

`w` here is **real compensation per employee derived from national accounts**, not the OECD average annual wage series. The two concepts are kept separate.


In [ ]:
from pathlib import Path
import pandas as pd

from wage_transmission.decomposition import decompose_panel
from wage_transmission.plots import plot_cumulative_decomposition, plot_decomposition_components


In [ ]:
input_path = Path('../data/processed/decomposition_inputs.csv')
if not input_path.exists():
    raise FileNotFoundError(
        'Run `poetry run wage-transmission download-decomposition` first.'
    )
panel = pd.read_csv(input_path)
panel.groupby('country').agg(first_year=('year', 'min'), last_year=('year', 'max'), n=('year', 'size'))


In [ ]:
portugal = panel.loc[panel['country'].eq('PRT')].copy()
components, summaries = decompose_panel(portugal)
summaries[0]


In [ ]:
components[[
    'year',
    'observed_real_wage_growth',
    'real_gdp_component',
    'labour_share_component',
    'employment_component',
    'relative_price_component',
    'identity_residual',
]].tail(10)


In [ ]:
figure_dir = Path('../results/decomposition/PRT')
plot_decomposition_components(components, figure_dir / 'annual_components.png')
plot_cumulative_decomposition(components, figure_dir / 'cumulative_components.png')


## Interpretation guardrail

The decomposition is an accounting identity, not a causal model. A negative labour-share contribution means the labour share moved against real compensation per employee over the interval; it does **not** by itself identify why the labour share changed.
